# 🤖 Chatbot Learning Buddy dengan Groq LLM dan MongoDB Atlas

Notebook ini berisi implementasi chatbot yang dapat:
1. Menjawab pertanyaan umum
2. Menjawab pertanyaan berdasarkan data dari MongoDB Atlas
3. Mengingat konteks percakapan sebelumnya

## Teknologi yang Digunakan:
- **Groq LLM**: Model bahasa untuk generate jawaban
- **LangChain**: Framework untuk RAG (Retrieval Augmented Generation)
- **ChromaDB**: Vector database untuk menyimpan embeddings
- **MongoDB Atlas**: Database sumber data
- **HuggingFace Embeddings**: Model untuk mengubah teks menjadi vector

## 1. Konfigurasi dan Setup

In [1]:
import os
from dotenv import load_dotenv

# Path ke file .env di folder frontend
ENV_PATH = r"C:\\Asah Dicoding\\learning-buddy\\backend\\.env"

# Load variabel dari .env
load_dotenv(ENV_PATH)

# Konfigurasi MongoDB Atlas
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("DB_NAME")

# Ambil list key Groq dari env (dipisah koma)
GROQ_API_KEYS = os.getenv("GROQ_API_KEYS", "").split(",")
GROQ_API_KEYS = [key.strip() for key in GROQ_API_KEYS if key.strip()]
# Variabel global untuk melacak key mana yang sedang dipakai
current_key_index = 0
# Groq API key

# Nama folder penyimpanan Chroma
CHROMA_COLLECTION_NAME = "learningbuddy_vector_db"

print("✅ Konfigurasi berhasil dimuat!")
print(f"📊 Database: {DB_NAME}")
print(f"💾 Vector DB: {CHROMA_COLLECTION_NAME}")

✅ Konfigurasi berhasil dimuat!
📊 Database: learning_buddy_db
💾 Vector DB: learningbuddy_vector_db


In [2]:
def get_current_api_key():
    """Mengambil API Key yang aktif saat ini"""
    global current_key_index
    return GROQ_API_KEYS[current_key_index]

def switch_api_key():
    """Pindah ke API Key berikutnya dalam list"""
    global current_key_index
    current_key_index = (current_key_index + 1) % len(GROQ_API_KEYS)
    new_key = GROQ_API_KEYS[current_key_index]
    print(f"\n⚠️ Rate Limit Terdeteksi! Mengganti ke API Key ke-{current_key_index + 1}...")
    return new_key

## 2. Mengambil Data dari MongoDB Atlas

**Catatan**: Jalankan cell ini hanya jika Anda belum membuat vector database atau ingin memperbarui data.

In [3]:
from pymongo import MongoClient
import json

try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    # Test koneksi
    client.server_info()
    
    db = client[DB_NAME]
    all_docs = []
    
    unique_course_set = set()
    unique_course_docs = []

    for collection_name in db.list_collection_names():
        collection = db[collection_name]
        docs = list(collection.find())

        for doc in docs:
            doc.pop("_id", None)  
            
            # 1. PROSES STANDAR: Simpan data asli apa adanya
            doc_original = doc.copy()
            doc_original["_collection"] = collection_name
            all_docs.append(json.dumps(doc_original, ensure_ascii=False))
            
            if collection_name == "LP+Course":
                lp_name = doc.get('learning_path_name')
                c_name = doc.get('course_name')
                c_level = doc.get('course_level_str')
                
                signature = (lp_name, c_name, c_level)
                
                if signature not in unique_course_set:
                    unique_course_set.add(signature)
                    
                    new_doc = {
                        "learning_path_name": lp_name,
                        "course_name": c_name,
                        "course_level_str": c_level,
                        "_collection": "Unique_Course" 
                    }
                    
                    unique_course_docs.append(json.dumps(new_doc, ensure_ascii=False))

    all_docs.extend(unique_course_docs)

    print(f"✅ Berhasil mengambil total {len(all_docs)} dokumen.")
    print(f"📚 Collections: {', '.join(db.list_collection_names())} ")
    
except Exception as e:
    print(f"❌ Error koneksi MongoDB: {str(e)}")

✅ Berhasil mengambil total 23383 dokumen.
📚 Collections: current_interest_questions, Learning_Path_Answer, modul, LP+Course, Course, Soal_Ujian, Learning_Path, data, Student_Progress, student_progress, current_tech_questions, users, Skill_Keywords, Tutorials, Course_Level 


In [4]:
import json

if 'all_docs' not in locals() or not all_docs:
    print("❌ Data 'all_docs' belum ditemukan. Harap jalankan cell 'Mengambil Data dari MongoDB' (Cell 5) terlebih dahulu.")
else:
    print(f"✅ Total Dokumen Tersimpan: {len(all_docs)}\n")

    found_collections = set()
    samples = {}
    doc_counts = {}

    print("⏳ Sedang menganalisis struktur data...")
    for doc_str in all_docs:
        # Kembalikan dari string JSON ke Dictionary (Object Python)
        doc_dict = json.loads(doc_str)
        
        col_name = doc_dict.get('_collection', 'Unknown')
        
        doc_counts[col_name] = doc_counts.get(col_name, 0) + 1
        
        if col_name not in found_collections:
            samples[col_name] = doc_dict
            found_collections.add(col_name)


    print("\n RINGKASAN DATA PER COLLECTION:")
    print(f"{'Nama Collection':<35} | {'Jml Dokumen'}")
    print("-" * 55)
    for col, count in doc_counts.items():
        print(f"{col:<35} | {count}")
    

    print("\n\n CONTOH STRUKTUR DATA (SAMPEL):")
    for col, sample_data in samples.items():
        print(f"\n{'='*80}")
        print(f"📂 Collection: {col}")
        print(f"{'-'*80}")
        
        print(f"🔑 Fields (Kolom): {list(sample_data.keys())}")
        
        print("\n📝 Contoh Isi:")
        print(json.dumps(sample_data, indent=2, ensure_ascii=False))

✅ Total Dokumen Tersimpan: 23383

⏳ Sedang menganalisis struktur data...

 RINGKASAN DATA PER COLLECTION:
Nama Collection                     | Jml Dokumen
-------------------------------------------------------
current_interest_questions          | 40
Learning_Path_Answer                | 67
LP+Course                           | 7916
Course                              | 75
Soal_Ujian                          | 1125
Learning_Path                       | 13
Student_Progress                    | 271
student_progress                    | 74
current_tech_questions              | 390
users                               | 8
Skill_Keywords                      | 5553
Tutorials                           | 7771
Course_Level                        | 5
Unique_Course                       | 75


 CONTOH STRUKTUR DATA (SAMPEL):

📂 Collection: current_interest_questions
--------------------------------------------------------------------------------
🔑 Fields (Kolom): ['question_desc', 'option_text'

## 3. Membuat Vector Database dengan ChromaDB

Proses ini akan:
1. Memecah dokumen menjadi chunks kecil
2. Mengubah chunks menjadi embeddings (vectors)
3. Menyimpan ke ChromaDB

In [5]:
import json
import shutil
import os
from langchain_huggingface import HuggingFaceEmbeddings # Pastikan import ini ada
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma


print("🔄 Sedang memformat data dengan Metadata...")

texts = []
metadatas = []
samples_per_source = {} 

for doc_str in all_docs:
    try:
        item = json.loads(doc_str)
        
        collection_name = item.pop('_collection', 'Umum') 
        item.pop('_id', None)
        
        content_parts = []
        for key, value in item.items():
            if value:
                clean_key = key.replace('_', ' ').title()
                content_parts.append(f"{clean_key}: {str(value)}")
        
        text_content = "\n".join(content_parts)
        
        texts.append(text_content)
        metadatas.append({"source": collection_name})
        
        # --- LOGIKA SAMPEL (Opsional, untuk melihat data) ---
        if collection_name not in samples_per_source:
            samples_per_source[collection_name] = []
        if len(samples_per_source[collection_name]) < 1: 
            samples_per_source[collection_name].append(text_content)
        
    except json.JSONDecodeError:
        continue

print(f"✅ Berhasil memproses {len(texts)} dokumen.")
print(f"📝 Contoh data 'users': {samples_per_source.get('users', ['Tidak ada data'])[0][:100]}...")


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150
)

docs = text_splitter.create_documents(texts, metadatas=metadatas)
print(f"📄 Total chunks: {len(docs)}")


print("⏳ Menyiapkan model embedding...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("⏳ Menyimpan ke ChromaDB...")
if os.path.exists(CHROMA_COLLECTION_NAME):
    shutil.rmtree(CHROMA_COLLECTION_NAME) 

vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=CHROMA_COLLECTION_NAME
)

print(f" Vector database berhasil dibuat dengan filter metadata!")

c:\Asah Dicoding\learning-buddy\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔄 Sedang memformat data dengan Metadata...
✅ Berhasil memproses 23372 dokumen.
📝 Contoh data 'users': Name: Angelio Asa Triatmaja
Email: ofurosasuke@gmail.com
Password: Angelio123
Created At: 2025-11-24...
📄 Total chunks: 23637
⏳ Menyiapkan model embedding...
⏳ Menyimpan ke ChromaDB...
 Vector database berhasil dibuat dengan filter metadata!


## 4. Inisialisasi Groq LLM

Groq menyediakan akses cepat ke berbagai model LLM open-source.

In [5]:
from langchain_groq import ChatGroq


def create_llm():
    """Fungsi untuk membuat objek LLM dengan key yang aktif"""
    api_key = get_current_api_key()
    return ChatGroq(
        groq_api_key=api_key,
        model_name="llama-3.3-70b-versatile",
        temperature=0.7,
        max_tokens=3000,
        timeout=30,
    )

llm = create_llm()
print(f"✅ Groq LLM berhasil diinisialisasi dengan Key indeks ke-{current_key_index}")

# Test LLM
print("\n🧪 Testing LLM...")
test_response = llm.invoke("Halo! Jawab dengan singkat: siapa kamu?")
print(f"Response: {test_response.content}")

c:\Asah Dicoding\learning-buddy\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Groq LLM berhasil diinisialisasi dengan Key indeks ke-0

🧪 Testing LLM...
Response: Halo! Saya adalah asisten AI, siap membantu Anda dengan informasi dan pertanyaan.


In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Definisikan Logic Routing dengan Konteks History
router_template = """
Anda adalah sistem klasifikasi niat (intent classifier) untuk chatbot edukasi bernama 'Learning Buddy'. 
Tugas Anda adalah menganalisis pertanyaan pengguna dan mengategorikannya ke dalam SATU kategori yang paling tepat.

PENTING: Perhatikan RIWAYAT PERCAKAPAN untuk memahami konteks jawaban singkat (seperti "A", "B", "Ya", "Tidak").

ANALISIS KATEGORI:

1. 'COURSE_INFO'
   - TERMASUK: Pertanyaan spesifik tentang satu learning path, detail tentang daftar course yang terdapat dalam learning path, seperti harga, durasi, atau materi dalam course tersebut. Selain itu dapat memberikan daftar course sesuai dengan learning path yang diminta.
   - CONTOH: "Apa materi dalam kelas Python?", "Berapa harga kelas Android?", "Apa saja urutan belajar menjadi Data Scientist?"

2. 'LEARNING_PATH'
   - TERMASUK: Pertanyaan tentang alur belajar (roadmap), atau daftar learning path yang ada dalam satu course ini.
   - CONTOH: "Learning path apa yang tersedia?"

3. 'PROGRESS'
   - TERMASUK: Pertanyaan tentang data pribadi siswa, nilai ujian, status kelulusan, sertifikat.
   - CONTOH: "Berapa nilai ujian saya?", "Apakah saya sudah lulus?"

4. 'SKILL'
   - TERMASUK: Pertanyaan tentang kemampuan/skill siswa, atau definisi teknis singkat.
   - CONTOH: "Skill apa saja yang saya miliki?", "Apa itu Python?"

5. 'RECOMMENDATION'
   - TERMASUK: Pengguna meminta saran, mencari rekomendasi, ATAU sedang menjawab pertanyaan interview dari Bot.
   - ATURAN KHUSUS: Jika dalam riwayat percakapan Bot sedang memberikan pertanyaan pilihan ganda (A/B/C) atau pertanyaan minat, dan User menjawab singkat (seperti "A", "B", "Web", "Mobile"), MASUKKAN KE KATEGORI INI.
   - CONTOH: "Saya bingung mau belajar apa", "A", "B", "Saya pilih opsi pertama".

6. 'GENERAL'
   - TERMASUK: Sapaan, pertanyaan di luar topik edukasi.
   - CONTOH: "Halo", "Selamat pagi".

---
RIWAYAT PERCAKAPAN (PENTING UNTUK KONTEKS):
{chat_history}
---

Pertanyaan Terakhir User: {question}

Kategori:"""

router_prompt = PromptTemplate.from_template(router_template)
# router_chain = router_prompt | llm | StrOutputParser()

def classify_question(question, history_str=""):
    """
    Menentukan kategori pertanyaan.
    PENTING: Tidak menggunakan try-except untuk Rate Limit agar bisa ditangkap fungsi chat.
    """
    # Gunakan variabel global llm yang terbaru
    global llm 
    
    # Buat chain secara dinamis agar selalu menggunakan key LLM yang aktif
    dynamic_router_chain = router_prompt | llm | StrOutputParser()
    
    try:
        category = dynamic_router_chain.invoke({
            "question": question, 
            "chat_history": history_str if history_str else "Tidak ada riwayat."
        }).strip()
        
        clean_category = category.replace("Kategori:", "").replace(".", "").strip()
        
        valid_keys = ['COURSE_INFO', 'PROGRESS', 'SKILL', 'RECOMMENDATION', 'GENERAL', 'LEARNING_PATH']
        for key in valid_keys:
            if key in clean_category:
                return key
        return 'GENERAL'
        
    except Exception as e:
        # Jika errornya Rate Limit, kita lempar (raise) ke atas agar fungsi chat tau
        if "429" in str(e) or "rate_limit_exceeded" in str(e):
            raise e 
        print(f"Router Error (Non-Limit): {e}")
        return 'GENERAL'

# --- TEST ROUTING ---
print("--- Berhasil ---")


--- Berhasil ---


## 5. Setup Retriever

Retriever akan mencari dokumen yang paling relevan dengan pertanyaan pengguna.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_chroma import Chroma
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

vectordb = Chroma(
    persist_directory=CHROMA_COLLECTION_NAME,
    embedding_function=embedding_model
)



## 6. Membuat Chatbot dengan Memory

Chatbot ini akan:
- Mengingat percakapan sebelumnya
- Menggunakan RAG untuk menjawab berdasarkan data MongoDB
- Menjawab pertanyaan umum dengan LLM

In [ ]:

# from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
import time
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)


def chat(question):
    global llm 
    
    print(f"\n{'='*60}")
    print(f"🤔 User: {question}")
    
    history_data = memory.load_memory_variables({})
    chat_history_list = history_data.get('chat_history', [])
    
    history_str = ""
    for msg in chat_history_list:
        role = "User" if msg.type == "human" else "Bot"
        history_str += f"{role}: {msg.content}\n"

    max_retries = len(GROQ_API_KEYS)
    attempt = 0
    
    while attempt < max_retries:
        try:
            category = classify_question(question, history_str)
            
            if attempt == 0 or category != 'GENERAL': 
                print(f"🧭 Konteks Terdeteksi: [{category}]")

            search_kwargs = {"k": 20}
            
            if category == 'COURSE_INFO':
                search_kwargs["filter"] = {"source": {"$in": ["Course", "Unique_Course"]}}
                system_instruction = "Jawab seputar kurikulum. Gunakan data course."
                
            elif category == 'LEARNING_PATH':
                search_kwargs["filter"] = {"source": {"$in": ["Learning_Path"]}}
                system_instruction = "Jawab seputar alur belajar."

            elif category == 'PROGRESS':
                search_kwargs["filter"] = {"source": {"$in": ["users", "student_progress"]}}
                system_instruction = "Anda admin akademik. Cek nilai/kelulusan siswa."

            elif category == 'SKILL':
                search_kwargs["filter"] = {"source": {"$in": ["users", "student_progress", "Skill_Keywords"]}}
                system_instruction = "Analisis skill user."

            elif category == 'RECOMMENDATION':
                search_kwargs["filter"] = {"source": {"$in": ["current_tech_questions", "current_interest_questions", "users", "student_progress", "Learning_Path", "Unique_Course"]}}
                system_instruction = """Anda adalah Konsultan Pendidikan Akademik. Tugas Anda adalah membimbing user menemukan jalur belajar yang tepat.
                    
                    IKUTI ALUR (FLOW) BERIKUT SECARA BERURUTAN BERDASARKAN RIWAYAT CHAT:

                    PHASE 1: CEK STATUS (Jika user baru menyapa/meminta rekomendasi awal)
                    - Cek data 'Student_Progress'.
                    - Jika user sedang aktif belajar, tanya: "Saya lihat progresmu di [Nama Course] masih berjalan. Apakah ingin melanjutkan itu atau mau belajar hal baru?"
                    
                    PHASE 2: VALIDASI MINAT (Jika user menjawab ingin hal baru)
                    - Tanya: "Apakah kamu sudah kepikiran mau belajar topik tertentu? (Misal: Android, AI, Web)"
                    
                    PHASE 3A: DIRECT RECOMMENDATION (Jika user menjawab SUDAH punya ide/topik)
                    - Langsung cari data di 'Learning_Path' yang cocok dengan topik tersebut.
                    - Berikan rekomendasi learning path yang tersedia beserta alasannya.
                    
                    PHASE 3B: INTERVIEW/ASSESSMENT (Jika user menjawab BELUM/TIDAK tahu)
                    - Katakan: "Baik, mari kita cari tahu minatmu lewat beberapa pertanyaan singkat."
                    - Ajukan pertanyaan dari 'current_interest_questions' atau 'current_tech_questions'.
                    - ATURAN INTERVIEW:
                    1. Ajukan HANYA SATU pertanyaan per respons. Tunggu user menjawab.
                    2. Cek 'RIWAYAT PERCAKAPAN'. Jangan ulangi pertanyaan yang sudah diajukan sebelumnya.
                    3. Lakukan ini sampai sekitar 5-7 pertanyaan terjawab.
                    
                    PHASE 4: FINAL RESULT (Setelah 5-7 pertanyaan terjawab)
                    - Analisis semua jawaban user di riwayat chat.
                    - Rekomendasikan 'Learning_Path' yang paling sesuai dengan jawaban user dari database.
                    """

            else: 
                search_kwargs["filter"] = {"source": {"$nin": ["Student_Progress", "users", "Learning_Path_Answer"]}}
                system_instruction = "Jawab secara umum dan ramah."

            specific_retriever = vectordb.as_retriever(
                search_type="similarity",
                search_kwargs=search_kwargs
            )
            
            docs = specific_retriever.invoke(question)
            
            if not docs and category != 'GENERAL':
                print("⚠️ Info spesifik tidak ada, mencari data umum...")
                docs = vectordb.as_retriever(search_kwargs={"k": 10}).invoke(question)
                
            context_text = "\n\n".join([d.page_content for d in docs])
            
            final_prompt = f"""Anda adalah Learning Buddy.
            INSTRUKSI KHUSUS: {system_instruction}
            RIWAYAT PERCAKAPAN SEBELUMNYA:
            {history_str}
            KONTEKS DATA DARI DATABASE (RAG):
            {context_text}
            PERTANYAAN BARU USER: {question}
            JAWABAN:"""
            
            response = llm.invoke(final_prompt)
            answer_text = response.content
            
            print(f"{'-'*60}")
            print(f"🤖 Bot:\n{answer_text}")
            print(f"{'='*60}\n")
            
            memory.save_context({"question": question}, {"answer": answer_text})
            
            return answer_text 
            
        except Exception as e:
            error_msg = str(e)
            
            if "429" in error_msg or "rate_limit_exceeded" in error_msg:
                print(f"❌ Percobaan {attempt + 1} Gagal: Rate Limit.")
                
                # ROTASI KUNCI API
                switch_api_key()     
                llm = create_llm()   
                
                attempt += 1
                time.sleep(1)
                print("🔄 Mencoba lagi dengan API Key baru...")
                continue 
            else:
                print(f"❌ Error non-limit: {e}")
                return "Maaf, terjadi kesalahan sistem."

    return "Maaf, semua server sibuk (Rate Limit). Silakan coba lagi nanti."

def clear_history():
    """Hapus riwayat percakapan"""
    memory.clear()
    print("🗑️ Riwayat percakapan telah dihapus!")


def show_history():
    """Tampilkan riwayat percakapan"""
    history = memory.load_memory_variables({})
    messages = history.get('chat_history', [])
    
    if not messages:
        print("📭 Belum ada riwayat percakapan")
        return
    
    print(f"\n{'='*80}")
    print("📜 RIWAYAT PERCAKAPAN")
    print(f"{'='*80}\n")
    
    for i, msg in enumerate(messages):
        role = "👤 User" if msg.type == "human" else "🤖 Bot"
        print(f"{role}: {msg.content}\n")
    
    print(f"{'='*80}\n")


print("✅ Fungsi helper berhasil dimuat!")
print("\n📝 Fungsi yang tersedia:")
print("   - chat(question): Tanya chatbot")
print("   - clear_history(): Hapus riwayat percakapan")
print("   - show_history(): Lihat riwayat percakapan")

✅ Fungsi helper berhasil dimuat!

📝 Fungsi yang tersedia:
   - chat(question): Tanya chatbot
   - clear_history(): Hapus riwayat percakapan
   - show_history(): Lihat riwayat percakapan


C:\Users\andika arsana\AppData\Local\Temp\ipykernel_10268\701173388.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


## 7. Contoh Penggunaan Chatbot


In [ ]:
show_history()

In [9]:
chat("Sebutkan course yang ada pada learning path React Developer")


🤔 User: Sebutkan course yang ada pada learning path React Developer
🧭 Konteks Terdeteksi: [COURSE_INFO]
------------------------------------------------------------
🤖 Bot:
Pada Learning Path React Developer, terdapat beberapa course yang dapat dipelajari, yaitu:

1. Belajar Dasar Pemrograman Web (Dasar)
2. Belajar Membuat Front-End Web untuk Pemula (Pemula)
3. Belajar Fundamental Aplikasi Web dengan React (Menengah)
4. Belajar Membuat Aplikasi Web dengan React (Pemula)
5. Belajar Dasar Pemrograman JavaScript (Dasar)
6. Menjadi React Web Developer Expert (Mahir)

Itulah beberapa course yang ada pada Learning Path React Developer.



'Pada Learning Path React Developer, terdapat beberapa course yang dapat dipelajari, yaitu:\n\n1. Belajar Dasar Pemrograman Web (Dasar)\n2. Belajar Membuat Front-End Web untuk Pemula (Pemula)\n3. Belajar Fundamental Aplikasi Web dengan React (Menengah)\n4. Belajar Membuat Aplikasi Web dengan React (Pemula)\n5. Belajar Dasar Pemrograman JavaScript (Dasar)\n6. Menjadi React Web Developer Expert (Mahir)\n\nItulah beberapa course yang ada pada Learning Path React Developer.'